# Hanoi Traffic Congestion Prediction

## Project Overview

Predict congestion ratio (travel time / free-flow time) for 40 routes in Hanoi using TomTom API data. Features include time of day, weather, route characteristics, and recent traffic history.

**Key results:**
- XGBoost R²: 0.97 (30-min prediction with recent history)
- XGBoost R²: 0.67 (without recent history)
- MAE: 0.025 (approx 1.5 minutes error for a 60-minute trip)

**Data:** 3,300 observations, 40 routes, 3 days, 30-min intervals

## 1. Setup

In [54]:
!pip install optuna

import optuna
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sqlalchemy import create_engine
from google.colab import userdata

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Suppress some warnings for cleaner output
tf.get_logger().setLevel('ERROR')

In [55]:
DB_URL = userdata.get('SUPABASE_DB_URL')

engine = create_engine(DB_URL)

query = """
SELECT
    t.route_id,
    t.congestion_ratio,
    t.observed_time,
    t.travel_time_seconds,
    t.no_traffic_time_seconds,
    EXTRACT(HOUR FROM t.observed_time) as hour,
    EXTRACT(MINUTE FROM t.observed_time) as minute,
    EXTRACT(DOW FROM t.observed_time) as day_of_week,
    EXTRACT(DAY FROM t.observed_time) as date,
    rg.length_meters,
    rg.straightness_ratio,
    w.temperature_celsius,
    w.weather_condition,
    CASE WHEN h.holiday_date IS NOT NULL THEN 1 ELSE 0 END as is_holiday,
    CASE WHEN t.is_rush_hour IS TRUE THEN 1 ELSE 0 END as is_rush_hour
FROM traffic_observations t
JOIN routes r ON t.route_id = r.route_id
LEFT JOIN route_geometries rg ON r.route_id = rg.route_id
LEFT JOIN weather_data w ON DATE(t.observed_time) = w.observation_date
    AND EXTRACT(HOUR FROM t.observed_time) = w.hour
LEFT JOIN holidays h ON DATE(t.observed_time) = h.holiday_date
WHERE t.congestion_ratio IS NOT NULL
"""

df = pd.read_sql(query, engine).copy()

seed = 85

## 2. Feature Engineering

### 2.1 Time Features

In [56]:
# Cyclical encoding for hour (24-hour cycle)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Cyclical encoding for minute (60-minute cycle)
df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

# Cyclical encoding for day of week (7-day cycle)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Linear time feature (minutes since 6 AM)
df['minutes_elapsed'] = (df['hour'] - 6) * 60 + df['minute']
df['minutes_elapsed'] = df['minutes_elapsed'].clip(0, 780)
df['minutes_elapsed_norm'] = df['minutes_elapsed'] / 780

### 2.2 Weather Features

In [57]:
df['temperature_norm'] = (
    df['temperature_celsius'] - df['temperature_celsius'].min()) / (
    df['temperature_celsius'].max() - df['temperature_celsius'].min()
)

df['weather_encoded'] = LabelEncoder().fit_transform(df['weather_condition'])

### 2.3 Interaction Features

In [58]:
df['rush_hour_route'] = df['is_rush_hour'] * df['route_id']
df['hour_sin_length'] = df['hour_sin'] * df['length_meters']
df['hour_sin_dow'] = df['hour_sin'] * df['dow_sin']
df['hour_cos_dow'] = df['hour_cos'] * df['dow_cos']

### 2.4 Lag and Rolling Features

These features are computed within each (route, date) group to prevent cross-day leakage.

In [59]:
df = df.sort_values(['route_id', 'observed_time'])
congestion_by_route_and_date = df.groupby(['route_id', 'date'])['congestion_ratio']

df['congestion_lag_1'] = congestion_by_route_and_date.shift(1)
df['congestion_lag_2'] = congestion_by_route_and_date.shift(2)
df['congestion_rolling_mean_3'] = congestion_by_route_and_date.transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['congestion_rolling_std_3'] = congestion_by_route_and_date.transform(
    lambda x: x.rolling(3, min_periods=1).std().fillna(0)
)

# Drop rows without enough history
df = df.dropna(subset=['congestion_lag_2'])

### 2.5 Feature Selection

In [60]:
feature_columns = [
    # Time features (cyclical)
    'hour_sin', 'hour_cos',
    'minute_sin', 'minute_cos',
    'dow_sin', 'dow_cos',
    'minutes_elapsed',

    # Rush hour
    'is_rush_hour',

    # Weather
    'weather_encoded',
    'temperature_celsius',

    # Lag features
    'congestion_lag_1', 'congestion_lag_2',
    'congestion_rolling_mean_3',
    'congestion_rolling_std_3',

    # Route features
    'length_meters',
    'straightness_ratio',

    # Interaction features
    'rush_hour_route',
    'hour_sin_length',
    'hour_sin_dow',
    'hour_cos_dow',

    # Holiday
    'is_holiday'
]

# Target
target_column = 'congestion_ratio'

X = df[feature_columns]
y = df[target_column]

## 3. Model Training

### 3.1 XGBoost with Hyperparameter Tuning

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = xgb.XGBRegressor(**params, random_state=seed)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='r2').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

best_model = xgb.XGBRegressor(**study.best_params, random_state=seed)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R2: {r2_score(y_test, y_pred):.3f}")
print(f"Best parameters: {study.best_params}")

[I 2026-05-21 10:02:59,789] A new study created in memory with name: no-name-cd8a8a81-c8d7-4c98-abc3-fd9c34d58668
[I 2026-05-21 10:03:31,842] Trial 0 finished with value: 0.9045833282389277 and parameters: {'n_estimators': 577, 'max_depth': 9, 'learning_rate': 0.017152687632662367, 'subsample': 0.8240989261462379, 'colsample_bytree': 0.8183551856157257, 'reg_alpha': 1.5033592706426074e-07, 'reg_lambda': 0.09371624061357312, 'min_child_weight': 1}. Best is trial 0 with value: 0.9045833282389277.
[I 2026-05-21 10:03:36,650] Trial 1 finished with value: 0.9314897823439944 and parameters: {'n_estimators': 792, 'max_depth': 7, 'learning_rate': 0.015811080818577454, 'subsample': 0.9095098787621679, 'colsample_bytree': 0.7935052828857265, 'reg_alpha': 1.8403030827124774e-07, 'reg_lambda': 0.0007639923729130426, 'min_child_weight': 6}. Best is trial 1 with value: 0.9314897823439944.
[I 2026-05-21 10:03:39,231] Trial 2 finished with value: 0.9064841842921488 and parameters: {'n_estimators': 819

MAE: 0.019
R2: 0.983
Best parameters: {'n_estimators': 556, 'max_depth': 4, 'learning_rate': 0.07576634007924662, 'subsample': 0.6213632257343789, 'colsample_bytree': 0.9797609980210629, 'reg_alpha': 5.0726541158819316e-05, 'reg_lambda': 0.016333815708323318, 'min_child_weight': 6}


Optuna performs 50 trials of Bayesian optimisation to find the best XGBoost parameters.

The tuned XGBoost achieves R² = 0.966 on the test set. MAE = 0.025 means prediction error is approximately 2.5% of the free-flow travel time.

In [62]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance.head(10))

                      feature  importance
12  congestion_rolling_mean_3    0.406039
0                    hour_sin    0.103514
6             minutes_elapsed    0.067078
10           congestion_lag_1    0.061857
11           congestion_lag_2    0.060910
13   congestion_rolling_std_3    0.050756
15         straightness_ratio    0.039270
1                    hour_cos    0.034165
19               hour_cos_dow    0.030165
9         temperature_celsius    0.026482


Rolling mean of congestion (past 90 minutes) is the strongest predictor, followed by time of day (hour_sin). This confirms that recent traffic history dominates short-term predictions.

### 3.2 Without Rolling Features (Baseline Comparison)

This shows how much predictive power comes from recent traffic history.

In [63]:
features_without_rolling = [f for f in feature_columns if 'rolling' not in f]
X_train_no_rolling = X_train[features_without_rolling]

model_no_rolling = xgb.XGBRegressor(**study.best_params)
model_no_rolling.fit(X_train_no_rolling, y_train)
y_pred_no_rolling = model_no_rolling.predict(X_test[features_without_rolling])

importance_no_rolling = pd.DataFrame({
    'feature': X_train_no_rolling.columns,
    'importance': model_no_rolling.feature_importances_
}).sort_values('importance', ascending=False)

print(f"R² without rolling features: {r2_score(y_test, y_pred_no_rolling):.4f}")
print(importance_no_rolling.head(10))

R² without rolling features: 0.7667
                feature  importance
7          is_rush_hour    0.279775
10     congestion_lag_1    0.156802
1              hour_cos    0.076833
0              hour_sin    0.062733
6       minutes_elapsed    0.056551
4               dow_sin    0.044880
13   straightness_ratio    0.039061
17         hour_cos_dow    0.038566
9   temperature_celsius    0.028446
11     congestion_lag_2    0.027275


### 3.3 Time Series Cross-Validation

Using expanding window to prevent future data leakage.

In [64]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

for train_idx, val_idx in tscv.split(X):
    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(**study.best_params)
    model.fit(X_train_cv, y_train_cv)
    y_pred_cv = model.predict(X_val_cv)
    cv_scores.append(r2_score(y_val_cv, y_pred_cv))

print(f"Time series CV R²: mean={np.mean(cv_scores):.4f}, std={np.std(cv_scores):.4f}")

Time series CV R²: mean=0.8373, std=0.2651


The high variance across folds indicates that model performance depends on the specific time period. This is expected with only 3 days of data. With more data (14+ days), the standard deviation would decrease.

## 4. Data Stability Analysis

Traffic stability directly affects model performance. High stability means high R² is expected.

In [65]:
print(f"Congestion ratio variance: {df['congestion_ratio'].var():.4f}")
print(f"Congestion ratio range: {df['congestion_ratio'].min():.2f} - {df['congestion_ratio'].max():.2f}")
print(f"Congestion ratio std: {df['congestion_ratio'].std():.4f}")
print()

# Check variance across full day for each route
group_cols = ['route_id', 'date']
variance_by_day = df.groupby(group_cols)['congestion_ratio'].var()
print(f"Daily variance per route - mean: {variance_by_day.mean():.4f}")
print(f"Daily variance per route - std: {variance_by_day.std():.4f}")
print(f"Days with variance < 0.05: {(variance_by_day < 0.05).sum()} / {len(variance_by_day)}")
print()

# Check variance at same hour across different days
group_cols = ['route_id', 'hour']
variance_by_hour = df.groupby(group_cols)['congestion_ratio'].var()
print(f"Hourly variance across days - mean: {variance_by_hour.mean():.4f}")
print(f"Hours with variance < 0.05: {(variance_by_hour < 0.05).sum()} / {len(variance_by_hour)}")
print()

# Check how much congestion changes between consecutive observations
df = df.sort_values(['route_id', 'date', 'observed_time'])
df['congestion_change'] = df.groupby(['route_id', 'date'])['congestion_ratio'].diff().abs()
print(f"Mean absolute change between 30-min intervals: {df['congestion_change'].mean():.4f}")
print(f"Median absolute change: {df['congestion_change'].median():.4f}")
print(f"95th percentile change: {df['congestion_change'].quantile(0.95):.4f}")
print()

Congestion ratio variance: 0.0921
Congestion ratio range: 0.94 - 3.68
Congestion ratio std: 0.3034

Daily variance per route - mean: 0.0593
Daily variance per route - std: 0.0843
Days with variance < 0.05: 107 / 160

Hourly variance across days - mean: 0.0248
Hours with variance < 0.05: 493 / 600

Mean absolute change between 30-min intervals: 0.1102
Median absolute change: 0.0500
95th percentile change: 0.4200



These statistics confirm that traffic in this dataset is inherently stable. A simple "no change" baseline would achieve high accuracy. The model's R² = 0.97 reflects this stability, not necessarily sophisticated pattern discovery.

## Conclusion

**What works:** Short-term congestion prediction using recent traffic history is highly accurate (R² = 0.97).

**Limitations:** Without recent history, accuracy drops significantly (R² = 0.67). Traffic in this dataset is relatively stable, which inflates the R².

**Future improvements:** Collect more data (14+ days), add more routes, include night hours.

## LSTM Experiment (Archived)

I attempted an LSTM to learn temporal patterns directly from raw sequences.
With 3,300 observations, the LSTM achieved only R² = 0.52, compared to
XGBoost's 0.97. This confirms that for small tabular time series,
gradient boosting is more sample-efficient than deep learning.